In [2]:
import sys; sys.path.append("..")
import pandas as pd, numpy as np
from src.features import extract_features, get_host

df = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv", usecols=["URL", "label"])
df = df.drop_duplicates(subset="URL").copy()
df["phish"] = (df["label"] == 0).astype(int)   # 1 = phishing (positive class)
df["host"] = df["URL"].map(get_host)

feats = pd.DataFrame([extract_features(u) for u in df["URL"]], index=df.index)
data = pd.concat([df[["host", "phish"]], feats], axis=1)
print(data.shape)
print("Unique hosts:", data["host"].nunique())

(235370, 21)
Unique hosts: 219094


In [3]:
from sklearn.model_selection import GroupShuffleSplit

# Hosts that appear with both labels would confuse the model
conflict = data.groupby("host")["phish"].nunique()
conflict = conflict[conflict > 1].index
print("Hosts with both labels:", len(conflict))
data = data[~data["host"].isin(conflict)]

X = data.drop(columns=["host", "phish"])
y = data["phish"]
groups = data["host"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Phishing share train/test:", round(y_train.mean(), 3), round(y_test.mean(), 3))
print("Hosts shared between train and test:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))

Hosts with both labels: 58
Train: (188682, 19) Test: (46529, 19)
Phishing share train/test: 0.429 0.42
Hosts shared between train and test: 0


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

for name, m in {"Logistic Regression": LogisticRegression(max_iter=2000),
                "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=1, random_state=42)}.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print(name)
    print(classification_report(y_test, pred, target_names=["legitimate", "phishing"], digits=3))
    print(confusion_matrix(y_test, pred, labels=[1, 0]), "\n")

Logistic Regression
              precision    recall  f1-score   support

  legitimate      0.792     0.957     0.867     27010
    phishing      0.916     0.652     0.762     19519

    accuracy                          0.829     46529
   macro avg      0.854     0.804     0.814     46529
weighted avg      0.844     0.829     0.823     46529

[[12733  6786]
 [ 1174 25836]] 

Random Forest
              precision    recall  f1-score   support

  legitimate      0.798     0.952     0.868     27010
    phishing      0.909     0.667     0.770     19519

    accuracy                          0.832     46529
   macro avg      0.854     0.809     0.819     46529
weighted avg      0.845     0.832     0.827     46529

[[13027  6492]
 [ 1309 25701]] 



In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix

m = HistGradientBoostingClassifier(random_state=42)
m.fit(X_train, y_train)
pred = m.predict(X_test)

print("Hist Gradient Boosting")
print(classification_report(y_test, pred, target_names=["legitimate", "phishing"], digits=3))
print(confusion_matrix(y_test, pred, labels=[1, 0]), "\n")

Hist Gradient Boosting
              precision    recall  f1-score   support

  legitimate      0.802     0.964     0.876     27010
    phishing      0.930     0.671     0.780     19519

    accuracy                          0.841     46529
   macro avg      0.866     0.818     0.828     46529
weighted avg      0.856     0.841     0.835     46529

[[13104  6415]
 [  979 26031]] 



In [6]:
p = m.predict_proba(X_test)[:, 1]          # probability of phishing (class 1)
is_ph = (y_test == 1).values

rows = []
for t in np.arange(0.05, 0.96, 0.05):
    flagged = p >= t
    fn = int((is_ph & ~flagged).sum())
    fp = int((~is_ph & flagged).sum())
    rows.append({"Threshold": round(t, 2),
                 "Phishing recall": round(1 - fn / is_ph.sum(), 3),
                 "Legit flagged %": round(fp / (~is_ph).sum() * 100, 1),
                 "FN": fn, "FP": fp})
pd.DataFrame(rows).set_index("Threshold")

,Phishing recall,Legit flagged %,FN,FP
Threshold,,,,
0.05,0.998,97.6,30,26367
0.10,0.988,88.0,232,23767
0.15,0.975,81.9,481,22115
0.20,0.820,29.4,3510,7949
0.25,0.775,17.2,4399,4657
0.30,0.742,11.5,5040,3095
0.35,0.721,8.2,5451,2219
0.40,0.697,6.2,5915,1664
0.45,0.683,4.6,6182,1242


In [7]:
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier

groups_train = groups.iloc[train_idx]
gb = HistGradientBoostingClassifier(random_state=42)
oof = cross_val_predict(gb, X_train, y_train, cv=GroupKFold(n_splits=5),
                        groups=groups_train, method="predict_proba")[:, 1]
is_ph_tr = (y_train == 1).values

rows = []
for t in [0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]:
    fl = oof >= t
    rows.append({"Threshold": t,
                 "Phishing recall": round((fl & is_ph_tr).sum() / is_ph_tr.sum(), 3),
                 "Legit flagged %": round((fl & ~is_ph_tr).sum() / (~is_ph_tr).sum() * 100, 1)})
pd.DataFrame(rows).set_index("Threshold")

,Phishing recall,Legit flagged %
Threshold,,
0.20,0.828,29.8
0.25,0.764,17.3
0.30,0.730,11.3
0.35,0.709,7.8
0.40,0.695,6.1
0.45,0.683,4.7
0.50,0.667,3.6


In [8]:
import joblib, json, os
from sklearn.ensemble import HistGradientBoostingClassifier

os.makedirs("../models", exist_ok=True)
gb = HistGradientBoostingClassifier(random_state=42).fit(X_train, y_train)
joblib.dump(gb, "../models/url_model.joblib")

meta = {"features": list(X_train.columns), "low_max": 0.30, "high_min": 0.50}
with open("../models/url_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

p_te = gb.predict_proba(X_test)[:, 1]
print("Test recall at 0.5:", round(((p_te >= 0.5) & (y_test == 1)).sum() / (y_test == 1).sum(), 3))
print("Saved", len(meta["features"]), "features")

Test recall at 0.5: 0.671
Saved 19 features
